# Pagination and Filtering

Every list endpoint in the SDK returns an `AsyncPage[T]` (or `SyncPage[T]`) object.
This notebook demonstrates every pagination pattern:

1. Single page — inspect items and metadata
2. Auto-paging — `list_all()` fetches all pages automatically
3. Manual pagination — `next_page()` for fine-grained control
4. `async for` iteration over a page
5. Filtering and sorting

> **These notebooks are async-first.** They use `AsyncWorkflowClient` with top-level `await`,
> which runs directly in Jupyter (the setup cell calls `nest_asyncio.apply()`). Every method
> shown also exists on the synchronous `WorkflowClient` — just drop the `await`. See the
> [docs](../docs/README.md) for the sync surface. Notebook bodies stay 100% async — there is
> no per-notebook sync cell.

In [ ]:
import _bootstrap  # noqa: F401 - enables import interactly (no install needed)

import os, sys
import nest_asyncio
from dotenv import load_dotenv
from pathlib import Path

# This is required to run asyncio in Jupyter Notebook
nest_asyncio.apply()

# Get the current notebook directory and find the project root
current_dir = Path(os.getcwd())
project_root = current_dir
while project_root.parent != project_root:
    if (project_root / '.env').exists():
        break
    project_root = project_root.parent
else:
    project_root = current_dir
    for _ in range(5):
        if (project_root / 'pyproject.toml').exists():
            break
        project_root = project_root.parent

# Load the environment variables from .env in project root
env_path = project_root / '.env'
if env_path.exists():
    load_dotenv(dotenv_path=str(env_path))
    print(f"Loaded .env from: {env_path}")
else:
    print(f"Warning: .env file not found at {env_path}")

# Add the project root to sys.path so imports resolve
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))
    print(f"Added to Python path: {project_root}")
else:
    print(f"Project root already in Python path: {project_root}")

In [ ]:
# Interactly credentials are read from environment variables.
#
# Convenience defaults point at the dev "Workflow Illustrations" org; your shell
# environment always wins (setdefault only fills in what you have not set).
os.environ.setdefault("INTERACTLY_BASE_URL", "https://api-dev.interactly.ai/workflows")
os.environ.setdefault("INTERACTLY_TEAM_ID", "67458e762b7d3dc15aaea5b5")
os.environ.setdefault("INTERACTLY_USER_ID", "687b1a4f745c8e6806c98d91")

# The bearer token is a secret — never hardcode it in the notebook.
# Export it before launching Jupyter:  export INTERACTLY_API_KEY="…"
assert os.environ.get("INTERACTLY_API_KEY"), (
    "Set INTERACTLY_API_KEY in your environment before running this notebook."
)

#print(f"API KEY is: {os.getenv('INTERACTLY_API_KEY')}")
print(f"TEAM ID is: {os.getenv('INTERACTLY_TEAM_ID')}")
print(f"USER ID is: {os.getenv('INTERACTLY_USER_ID')}")
print(f"BASE URL is: {os.getenv('INTERACTLY_BASE_URL')}")

In [ ]:
import _bootstrap  # noqa: F401 - enables import interactly (no install needed)

import nest_asyncio

from interactly import AsyncWorkflowClient

# Required to run top-level `await` inside a Jupyter notebook
nest_asyncio.apply()

client = AsyncWorkflowClient()
print("Connected to", client._base_url)

## 1. Single page — inspect metadata

In [ ]:
from interactly._pagination import AsyncPage
from interactly.types.workflows.workflow import Workflow

page: AsyncPage[Workflow] = await client.workflows.list(size=10)

print(f"Items on this page : {len(page.items)}")
print(f"Total items in team: {page.total}")
print(f"Has next page      : {page.has_next_page}")  # property, no parentheses

for wf in page.items:
    print(f"  {wf.id}  {wf.name!r}")

## 2. Auto-paging — fetch everything at once

`list_all()` transparently issues as many requests as needed and returns a flat list.

In [ ]:
from interactly._pagination import AsyncPage
from interactly.types.workflows.workflow import Workflow

page: AsyncPage[Workflow] = await client.workflows.list(size=25)
all_workflows = await page.list_all()

print(f"All workflows fetched: {len(all_workflows)}")

## 3. Manual pagination with `next_page()`

Useful when you want to stop early (e.g., process in batches, apply a search predicate).
`has_next_page` is a property; `next_page()` is awaited.

In [ ]:
PAGE_SIZE = 5

current_page = await client.workflows.list(size=PAGE_SIZE)
page_num = 1

while True:
    print(f"--- Page {page_num} ({len(current_page.items)} items) ---")
    for wf in current_page.items:
        print(f"  {wf.id}  {wf.name!r}")

    if not current_page.has_next_page:
        print("\nNo more pages.")
        break

    current_page = await current_page.next_page()
    page_num += 1

## 4. Paginating runs for a specific workflow

In [ ]:
# Replace with a workflow ID that has run history
WORKFLOW_ID = all_workflows[0].id if all_workflows else "<your-workflow-id>"

runs_page = await client.runs.list(workflow_id=WORKFLOW_ID, size=20)
all_runs = await runs_page.list_all()

print(f"Total runs for workflow {WORKFLOW_ID}: {len(all_runs)}")
for run in all_runs[:5]:
    print(f"  {run.id}  status={run.status}  started={run.started_at}")

## 5. `async for` iteration

An `AsyncPage` is itself async-iterable — `async for` walks the items of the
**current** page. Combine with `next_page()` (section 3) or `list_all()` (section 2)
to span multiple pages.

In [ ]:
from interactly._pagination import AsyncPage
from interactly.types.workflows.workflow import Workflow

page: AsyncPage[Workflow] = await client.workflows.list(size=10)
async for wf in page:
    print(f"  {wf.id}  {wf.name!r}")

## 6. Filtering and sorting

Most list endpoints accept query parameters for filtering. The exact parameters
vary by resource — see [`../docs/api_async.md`](../docs/api_async.md) for a full reference.

In [ ]:
# Filter runs by status
completed_page = await client.runs.list(
    workflow_id=WORKFLOW_ID,
    status="completed",
    size=50,
)
completed_runs = await completed_page.list_all()

print(f"Completed runs: {len(completed_runs)}")

## 7. Counting without fetching all items

The `total` field on the first page is returned by the server without fetching all records.

In [ ]:
# Fetch a tiny page just to get the total count
count_page = await client.workflows.list(size=1)
print(f"Total workflows in this team: {count_page.total}")

count_page = await client.runs.list(workflow_id=WORKFLOW_ID, size=1)
print(f"Total runs for workflow {WORKFLOW_ID}: {count_page.total}")